# AI for the Classroom

A subset of NRP's GPUs power a community-shared, **OpenAI-compatible LLM endpoint** at `https://ellm.nrp-nautilus.io/v1`. No pod to run, no GPU to hold, no per-seat license.

This notebook is a short taste of two things a class gets for free on this hub:

1. **An LLM you can call from Python** — the token is already in your environment.
2. **Jupyter AI** — an assistant living in JupyterLab that can explain and fix the code in front of you.

> 📘 [Managed LLMs](https://nrp.ai/documentation/userdocs/ai/llm-managed/) · [Available models](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) · [Get your own token](https://nrp.ai/llmtoken)

## 1. The token is already here

On this training hub, `OPENAI_API_BASE` and `OPENAI_API_KEY` are exported into every server for you — **students never handle a key**. On your own hub you would mint one at [nrp.ai/llmtoken](https://nrp.ai/llmtoken) and set the same two variables.

In [ ]:
import os

print("base:", os.environ.get("OPENAI_API_BASE", "(not set)"))
key = os.environ.get("OPENAI_API_KEY", "")
print("key: ", (key[:8] + "…") if key else "(not set)")

## 2. What models are on offer

One endpoint fronts the whole catalog — chat models large and small, a code model, and an embeddings model.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=os.environ["OPENAI_API_BASE"])

for m in client.models.list().data:
    print(m.id)

<details>
<summary>Expected output (the catalog rotates)</summary>

<pre style="line-height:1.45">
glm-5
qwen3
qwen3-embedding
qwen3-small
gemma-small-e4b
gpt-oss
gemma
kimi
minimax-m2
deepseek-v4-flash
gemma-small
gemma4-small
gemma4-12b
</pre>
</details>

Models come and go as the team rotates capacity, so check the [models page](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) before pinning a name into a syllabus — and prefer a small model for classwork, since it answers faster and leaves the big ones free.

> **What the model list does and does not prove.** On the NRP gateway it answers **without authentication**, so a successful listing proves only that the endpoint is reachable. The first call that actually checks your token is a chat completion — that is the real smoke test, and it is the next cell.

## 3. Ask it something

Picking one of those names is the only change needed; the `openai` SDK talks to NRP unchanged, because only `base_url` differs from the commercial API.

One wrinkle worth knowing: the SDK looks for `OPENAI_BASE_URL`, but NRP exports `OPENAI_API_BASE`, so pass it explicitly (as above) or the client quietly calls `api.openai.com` instead.

In [ ]:
resp = client.chat.completions.create(
    model="gpt-oss",
    messages=[
        {"role": "user",
         "content": "In two sentences, what is the National Research Platform?"},
    ],
)

print(resp.choices[0].message.content)

**That is the whole trick.** Those few lines work against a commercial API, against NRP, or against a vLLM server you run yourself on a GPU pod — you change `base_url` and nothing else. Teach the OpenAI-compatible API once and the skill outlives whatever model is fashionable this year.

## 4. A detour: notebook magics

A **magic** is a notebook command that is not Python. Line magics start with `%`, cell magics with `%%` and take the whole cell. They are worth ten minutes of any intro course, because they cover the things students otherwise ask you about one at a time.

In [ ]:
%who            # every variable you have defined so far
%whos           # …the same, with types and values

In [ ]:
%%time
# how long did this cell take?
total = sum(i * i for i in range(2_000_000))
print(total)

In [ ]:
%timeit sum(range(10_000))     # times a single statement, averaged over many runs

In [ ]:
# a "!" line runs in the shell, so the terminal is never far away
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU on this server"
!df -h /home/jovyan | tail -1

`%lsmagic` lists every magic available, and `%magic_name?` explains one. A handful worth knowing beyond the above:

| Magic | What it does |
|---|---|
| `%%writefile name.py` | dumps the cell to a file — how you hand students a script |
| `%run script.py` | runs a file in the current kernel, variables and all |
| `%matplotlib inline` | renders plots in the notebook |
| `%%bash` | runs the whole cell as a shell script |
| `%debug` | drops into a debugger at the last exception |
| `%load_ext` | loads an extension — which is exactly how the next section starts |

## 5. Jupyter AI

The hub ships [Jupyter AI](https://jupyter-ai.readthedocs.io/) already pointed at the NRP endpoint — no install, no key to paste.

- **Chat panel:** click the **chat icon** in the left sidebar and ask a question.
- **In a cell:** load the magics, below, and prefix a cell with `%%ai`.

> **The chat panel cannot see your notebook.** It has no access to the kernel, your variables, or your cell outputs — ask it *"why is 86.333 the average?"* and it will quite reasonably ask what you are talking about. `/learn` does not help either: it indexes **files on disk**, not the running kernel.
>
> The `%%ai` magics *can* reach into the kernel. That is what the rest of this section uses.

In [ ]:
%load_ext jupyter_ai_magics

Load that **before** you run anything that fails: it also installs the exception hook that `%ai error` reads from, and errors raised earlier are not recorded.

`%ai list` shows every model registered with Jupyter AI. Then a plain question, answered on NRP GPUs:

In [ ]:
%ai list

In [ ]:
%%ai openai-chat:gpt-oss
Explain what a Kubernetes namespace is, for someone who has never used one.

### Now break something on purpose

The three cells below are broken — each one names the bug it carries, since Jupyter AI will happily rewrite a cell in place or drop a corrected copy underneath it, and positions shift as soon as it does.

The first two crash. The third does not, and **that is the interesting one.**

In [ ]:
# 🐞 BROKEN — temperature conversion   (this one crashes)
temperatures_c = [18, 21, 25, 30, 12]

def to_fahrenheit(c):
    return c * 9 / 5 + 32

for t in temperatures_c:
    print(t, "C =", to_farenheit(t), "F")

In [ ]:
# 🐞 BROKEN — class average   (this one crashes)
student_scores = {"ana": 88, "ben": 92, "cleo": 79}

total = 0
for name in student_scores:
    total += name

print("class average:", total / len(student_scores))

In [ ]:
# 🐞 BROKEN — sensor average   (no crash; the answer is just wrong)
readings = [3, 7, 2, 9, 4, 8]

# intended: the average of every reading after the first
average = sum(readings[1:]) / len(readings)

print("average of readings 2..6 =", average)
print("expected:", (7 + 2 + 9 + 4 + 8) / 5)

### Asking about what just happened

Two things the magics do that the chat panel cannot.

**1. Explain the last error.** `%ai error` reaches into the kernel for the most recent traceback and explains it — nothing to copy or paste:

In [ ]:
%ai error openai-chat:gpt-oss

**2. Ask about your actual values.** Anything in `{curly braces}` inside a `%%ai` prompt is replaced with the *live value* of that variable from the kernel. This is the fix for "why is this number what it is?" — you hand it the number:

In [ ]:
%%ai openai-chat:gpt-oss
Here are some sensor readings: {readings}

My code computed an average of {average}, but what I wanted was the average of
every reading *after the first one*. Is {average} the right answer? Show the
arithmetic either way.

That prompt reaches the model with the real list and the real number already substituted in, which is why it can answer instead of asking you for context.

The **sensor average** cell is the one worth dwelling on in front of a class. It runs, prints a number, and the number is wrong — it divides by 6 when it should divide by 5. There is no traceback to paste, so the assistant has to reason about what the code was *meant* to do. That is the failure mode that quietly survives into a student's homework.

> **During the demo:** asked to fix a cell, Jupyter AI often offers more than one correction — a loop and a one-liner, say — and both will be right. Worth naming rather than glossing over: the assistant proposes, the student still reads the proposal and chooses.

## What to take away

- The endpoint is **OpenAI-compatible**, so every tool that speaks that API — notebooks, Jupyter AI, agents, your own scripts — works against NRP with a `base_url` change.
- On a hub you run, the token is an environment variable *you* set once, so **no student ever handles a credential**.
- An assistant sitting inside JupyterLab changes what a stuck student does at 2am.

Next: [deploy a JupyterHub of your own](3_custom_jupyterhub.html), with the image menu, resource limits and shared storage a course needs.